# 4.3 오버피팅과 정규화 (Overfitting and Regularization)
머신러닝 모델
과적합(Overfitting)

In [1]:
import torch # torch 임포트
import torch.nn as nn # nn 임포트
import torch.optim as optim # optim 임포트
import torch.nn.functional as F # functional 임포트
from torchvision import transforms, datasets # 비전 데이터 처리 임포트

In [2]:
USE_CUDA = torch.cuda.is_available() # GPU 가능 여부 확인
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 장치 설정

In [3]:
EPOCHS = 50 # 에포크 설정
BATCH_SIZE = 64 # 배치 크기 설정
DEVICE # 장치 출력

## 데이터셋에 노이즈 추가하기
![original.png](./assets/original.png)
![horizontalflip.png](./assets/horizontalflip.png)

In [4]:
train_loader = torch.utils.data.DataLoader( # 학습 로더 설정
    datasets.FashionMNIST('./.data', # 데이터 경로
                   train=True, # 학습 데이터
                   download=True, # 다운로드 여부
                   transform=transforms.Compose([ # 변환 설정
                       transforms.RandomHorizontalFlip(), # 랜덤 좌우 반전
                       transforms.ToTensor(), # 텐서 변환
                       transforms.Normalize((0.1307,), (0.3081,)) # 정규화
                   ])),
    batch_size=BATCH_SIZE, shuffle=True) # 배치 및 셔플 설정
test_loader = torch.utils.data.DataLoader( # 테스트 로더 설정
    datasets.FashionMNIST('./.data', # 데이터 경로
                   train=False, # 테스트 데이터
                   transform=transforms.Compose([ # 변환 설정
                       transforms.ToTensor(), # 텐서 변환
                       transforms.Normalize((0.1307,), (0.3081,)) # 정규화
                   ])),
    batch_size=BATCH_SIZE, shuffle=True) # 배치 및 셔플 설정

## 뉴럴넷으로 Fashion MNIST 학습하기
입력 `x` 는 `[배치크기, 색, 높이, 넓이]`로 이루어져 있습니다.
`x.size()`를 해보면 `[64, 1, 28, 28]`이라고 표시되는 것을 보실 수 있습니다.
Fashion MNIST에서 이미지의 크기는 28 x 28, 색은 흑백으로 1 가지 입니다.
그러므로 입력 x의 총 특성값 갯수는 28 x 28 x 1, 즉 784개 입니다.
우리가 사용할 모델은 3개의 레이어를 가진 뉴럴네트워크 입니다. 

In [5]:
class Net(nn.Module): # 신경망 클래스
    def __init__(self, dropout_p=0.2): # 초기화
        super(Net, self).__init__() # 부모 초기화
        self.fc1 = nn.Linear(784, 256) # 레이어 1
        self.fc2 = nn.Linear(256, 128) # 레이어 2
        self.fc3 = nn.Linear(128, 10) # 출력 레이어
        self.dropout_p = dropout_p # 드롭아웃 확률

    def forward(self, x): # 순전파
        x = x.view(-1, 784) # 평탄화
        x = F.relu(self.fc1(x)) # fc1 + ReLU
        x = F.dropout(x, training=self.training, # 드롭아웃
                      p=self.dropout_p) # 확률 적용
        x = F.relu(self.fc2(x)) # fc2 + ReLU
        x = F.dropout(x, training=self.training, # 드롭아웃
                      p=self.dropout_p) # 확률 적용
        x = self.fc3(x) # 출력층
        return x # 반환

## 모델 준비하기 
`to()` 함수는 모델의 파라미터들을 지정한 곳으로 보내는 역할을 합니다.
일반적으로 CPU 1개만 사용할 경우 필요는 없지만,
GPU를 사용하고자 하는 경우 `to("cuda")`로 지정하여 GPU로 보내야 합니다.
지정하지 않을 경우 계속 CPU에 남아 있게 되며 빠른 훈련의 이점을 누리실 수 없습니다.
최적화 알고리즘으로 파이토치에 내장되어 있는 `optim.SGD`를 사용하겠습니다.

In [ ]:
model        = Net(dropout_p=0.2).to(DEVICE) # 모델 생성 및 장치 이동
optimizer    = optim.SGD(model.parameters(), lr=0.01) # SGD 최적화기

## 학습하기

In [7]:
def train(model, train_loader, optimizer): # 학습 함수
    model.train() # 학습 모드 설정
    for batch_idx, (data, target) in enumerate(train_loader): # 배치 반복
        data, target = data.to(DEVICE), target.to(DEVICE) # 장치 이동
        optimizer.zero_grad() # 기울기 초기화
        output = model(data) # 예측
        loss = F.cross_entropy(output, target) # 손실 계산
        loss.backward() # 역전파
        optimizer.step() # 가중치 업데이트

In [8]:
def evaluate(model, test_loader): # 평가 함수
    model.eval() # 평가 모드 설정
    test_loss = 0 # 손실 합계
    correct = 0 # 정답 개수
    with torch.no_grad(): # 기울기 계산 비활성화
        for data, target in test_loader: # 배치 반복
            data, target = data.to(DEVICE), target.to(DEVICE) # 장치 이동
            output = model(data) # 예측
            test_loss += F.cross_entropy(output, target, # 손실 합산
                                         reduction='sum').item() # 합계 모드
            
            pred = output.max(1, keepdim=True)[1] # 예측 클래스 추출
            correct += pred.eq(target.view_as(pred)).sum().item() # 정답 확인

    test_loss /= len(test_loader.dataset) # 평균 손실
    test_accuracy = 100. * correct / len(test_loader.dataset) # 정확도
    return test_loss, test_accuracy # 반환

## 코드 돌려보기
자, 이제 모든 준비가 끝났습니다. 코드를 돌려서 실제로 훈련이 되는지 확인해봅시다!

In [ ]:
for epoch in range(1, EPOCHS + 1): # 에포크 반복
    train(model, train_loader, optimizer) # 학습
    test_loss, test_accuracy = evaluate(model, test_loader) # 평가
    
    print('[{}] Test Loss: {:.4f}, Accuracy: {:.2f}%'.format( # 로그 출력
          epoch, test_loss, test_accuracy))